In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")

GPU Available: True
Device Name: Tesla T4


**Installation & Dataset Setup**


In [ ]:
# Install Unsloth for ultra-fast QLoRA training
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

import json

# Synthetic DSP Anomaly -> Action Training Set
training_data = [
    {
        "instruction": "Analyze telemetry DSP metrics and determine corrective system action.",
        "input": "Peak Frequency: 320.0 Hz, RMS Power: 1.53, Anomaly Detected: True",
        "output": "DIAGNOSIS: High-frequency mechanical resonance & thermal vibration detected. ROOT CAUSE: Bearing wear or shaft misalignment. ACTION: Trigger circuit isolation, reduce duty cycle by 40%, and log error code E-402."
    },
    {
        "instruction": "Analyze telemetry DSP metrics and determine corrective system action.",
        "input": "Peak Frequency: 50.0 Hz, RMS Power: 0.72, Anomaly Detected: False",
        "output": "DIAGNOSIS: Nominal harmonic response. ROOT CAUSE: None. ACTION: Maintain standard operational status."
    },
    {
        "instruction": "Analyze telemetry DSP metrics and determine corrective system action.",
        "input": "Peak Frequency: 450.0 Hz, RMS Power: 2.10, Anomaly Detected: True",
        "output": "DIAGNOSIS: Severe electrical inverter switching noise. ROOT CAUSE: Inverter MOSFET switching fault. ACTION: Isolate primary power rail, switch to battery secondary, and trigger code E-901."
    }
]

with open("dsp_dataset.jsonl", "w") as f:
    for entry in training_data:
        f.write(json.dumps(entry) + "\n")

print("Dataset created successfully!")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-h4g2sy9i/unsloth_31852bc8d571494e9b2d4480dbf982f8
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-h4g2sy9i/unsloth_31852bc8d571494e9b2d4480dbf982f8
  Resolved https://github.com/unslothai/unsloth.git to commit 7c63bc8c4f18c1d0b0eec5e656ae20797893b500
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.0 MB/s eta 0:00:00
   ━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 59.8 MB/s eta 0:00:00
Dataset created successfully!


**QLoRA Model Training**

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

max_seq_length = 2048

# Load Llama-3.2-3B Instruct model in 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# Configure Target LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
)

# Load dataset
dataset = load_dataset("json", data_files="dsp_dataset.jsonl", split="train")

def format_prompts(examples):
    texts = []
    for inst, inp, out in zip(examples['instruction'], examples['input'], examples['output']):
        text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{inst}<|eot_id|><|start_header_id|>user<|end_header_id|>\n{inp}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n{out}<|eot_id|>"
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(format_prompts, batched = True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

trainer.train()

# Save LoRA Adapters
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("QLoRA Adapter Training Complete!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

...

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 30 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,4.854086
2,4.854075
3,4.633687
4,4.154266
5,3.659404
6,3.174684
7,2.620506
8,2.044770
9,1.506649
10,1.081146


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-30/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in lora_model/tokenizer_config.json.


QLoRA Adapter Training Complete!


**Test Inference & Download Weights**

In [ ]:
# 1. Test inference on the fine-tuned model
FastLanguageModel.for_inference(model)

inputs = tokenizer(
    [
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nAnalyze telemetry DSP metrics and determine corrective system action.<|eot_id|><|start_header_id|>user<|end_header_id|>\nPeak Frequency: 320.0 Hz, RMS Power: 1.53, Anomaly Detected: True<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    ],
    return_tensors="pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
print("--- Fine-Tuned Model Response ---")
print(tokenizer.batch_decode(outputs)[0])

# 2. Zip the lora_model directory for download
!zip -r lora_model.zip lora_model

# 3. Download to your local machine
from google.colab import files
files.download("lora_model.zip")

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Fine-Tuned Model Response ---
<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>
Analyze telemetry DSP metrics and determine corrective system action.<|eot_id|><|start_header_id|>user<|end_header_id|>
Peak Frequency: 320.0 Hz, RMS Power: 1.53, Anomaly Detected: True<|eot_id|><|start_header_id|>assistant<|end_header_id|>
DIAGNOSIS: High-frequency mechanical resonance & thermal vibration detected. ROOT CAUSE: Bearing wear or shaft misalignment. ACTION: Trigger circuit isolation, reduce duty cycle by 40%, and log error code E-402.<|eot_id|>
  adding: lora_model/ (stored 0%)
  adding: lora_model/chat_template.jinja (deflated 71%)
  adding: lora_model/adapter_config.json (deflated 59%)
  adding: lora_model/README.md (deflated 65%)
  adding: lora_model/tokenizer.json (deflated 85%)
  adding: lora_model/tokenizer_config.json (deflated 96%)
  adding: lora_model/adapter_model.safetensors (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>